# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

#### Business Problem

A used car dealership wants to know what makes a car worth more or less money. With that knowledge, they can buy the right cars and price them correctly.

We follow the **CRISP-DM** process (Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment) to work through this problem in a structured way.

#### Data Problem

We will build a **regression model** to predict `price` from car features like `year`, `odometer`, `condition`, `manufacturer`, `fuel`, `transmission`, `drive`, `type`, and `cylinders`.

Two goals:
1. Predict price accurately.
2. Identify which features matter most — so the dealership knows what to look for when buying inventory.

**How we measure success**: RMSE (Root Mean Squared Error). It tells us how far off our price predictions are, in dollars. A smaller RMSE means better predictions. We also track R², which shows what percentage of price variation our model explains.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

In [6]:
df = pd.read_csv('data/vehicles.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())

Shape: (426880, 18)

Columns: ['id', 'region', 'price', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'state']


In [7]:
df.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


In [8]:
pd.DataFrame({
    'dtype': df.dtypes,
    'nunique': df.nunique(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})

,dtype,nunique,null_count,null_pct
id,int64,426880,0,0.00
region,object,404,0,0.00
price,int64,15655,0,0.00
year,float64,114,1205,0.28
manufacturer,object,42,17646,4.13
model,object,29649,5277,1.24
condition,object,6,174104,40.79
cylinders,object,8,177678,41.62
fuel,object,5,3013,0.71
odometer,float64,104870,4400,1.03


In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,426880.0,7.311487e+09,4.473170e+06,7.207408e+09,7.308143e+09,7.312621e+09,7.315254e+09,7.317101e+09
price,426880.0,7.519903e+04,1.218228e+07,0.000000e+00,5.900000e+03,1.395000e+04,2.648575e+04,3.736929e+09
year,425675.0,2.011235e+03,9.452120e+00,1.900000e+03,2.008000e+03,2.013000e+03,2.017000e+03,2.022000e+03
odometer,422480.0,9.804333e+04,2.138815e+05,0.000000e+00,3.770400e+04,8.554800e+04,1.335425e+05,1.000000e+07


In [ ]:
# Visualize continuous variables - distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['price'].dropna(), bins=50, edgecolor='black')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(df['year'].dropna(), bins=30, edgecolor='black')
axes[1].set_title('Year Distribution')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')

axes[2].hist(df['odometer'].dropna(), bins=50, edgecolor='black')
axes[2].set_title('Odometer Distribution')
axes[2].set_xlabel('Miles')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize categorical variables
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

df['condition'].value_counts().plot(kind='bar', ax=axes[0, 0], edgecolor='black')
axes[0, 0].set_title('Condition Distribution')
axes[0, 0].set_xlabel('Condition')
axes[0, 0].set_ylabel('Count')

df['fuel'].value_counts().plot(kind='bar', ax=axes[0, 1], edgecolor='black')
axes[0, 1].set_title('Fuel Type Distribution')
axes[0, 1].set_xlabel('Fuel Type')
axes[0, 1].set_ylabel('Count')

df['transmission'].value_counts().plot(kind='bar', ax=axes[1, 0], edgecolor='black')
axes[1, 0].set_title('Transmission Distribution')
axes[1, 0].set_xlabel('Transmission')
axes[1, 0].set_ylabel('Count')

df['type'].value_counts().plot(kind='bar', ax=axes[1, 1], edgecolor='black')
axes[1, 1].set_title('Vehicle Type Distribution')
axes[1, 1].set_xlabel('Type')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### Data Understanding - Findings

**What we found:**

1. **Price (our target)**: Most cars cost between $5,900 and $26,485. But there are extreme outliers — some listed as $0, some over $3 billion. These are errors and will be removed.

2. **Year**: Most cars are from 2008-2017. This is good — the cars are recent enough to be useful for dealerships.

3. **Odometer (mileage)**: Most cars have been driven between 37,700 and 133,500 miles. But some show 10 million miles (clearly wrong). These will be removed.

4. **Condition**: Most cars are in "good" or "fair" condition. Only a few are in "excellent" condition. About 40% of this column is missing.

5. **Fuel type**: Almost all cars use gasoline. Very few use other fuels. This means fuel type might not be a strong price driver.

6. **Transmission**: Manual transmissions are rare. Most cars are automatic. 

7. **Vehicle type**: Most cars are sedans and trucks. Vans and convertibles are less common.

**Next steps:**
- Remove impossible data (price $0, odometer 10M miles)
- Decide what to do with missing values
- Prepare data for modeling

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.